# 03 — Feature Engineering

## Purpose

Create the final modeling dataset, document the selected features, inspect missingness, and export a reproducible processed CSV for modeling.

In [14]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Project root configured successfully.")

Project root configured successfully.


In [15]:
import pandas as pd

from config import RAW_DATA_DIR, PROCESSED_DATA_DIR, READMISSION_WINDOW_DAYS
from data_preparation import (
    load_raw_tables,
    parse_dates,
    build_readmission_dataset,
    get_model_features,
)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

tables = parse_dates(load_raw_tables(RAW_DATA_DIR))
df = build_readmission_dataset(tables, READMISSION_WINDOW_DAYS)

features, numeric_features, categorical_features = get_model_features(df)

print("Total model features:", len(features))
print("\nNumeric:")
print(numeric_features)
print("\nCategorical:")
print(categorical_features)

Total model features: 20

Numeric:
['age', 'length_of_stay_days', 'base_encounter_cost', 'total_claim_cost', 'payer_coverage', 'patient_cost', 'insurance_coverage_pct', 'procedure_count', 'prior_inpatient_visits', 'prior_readmissions', 'days_since_previous_discharge', 'days_since_previous_discharge_missing', 'admission_month', 'admission_dayofweek', 'weekend_admission']

Categorical:
['gender', 'race', 'ethnicity', 'marital', 'payer_name']


## Feature availability

In [16]:
feature_profile = pd.DataFrame({
    "feature": features,
    "dtype": [str(df[c].dtype) for c in features],
    "missing_count": [int(df[c].isna().sum()) for c in features],
    "missing_pct": [float(df[c].isna().mean() * 100) for c in features],
    "unique_values": [int(df[c].nunique(dropna=True)) for c in features],
}).sort_values("missing_pct", ascending=False)

feature_profile

,feature,dtype,missing_count,missing_pct,unique_values
10,days_since_previous_discharge,float64,153,13.527851,951
18,marital,str,1,0.088417,2
0,age,float64,0,0.000000,72
1,length_of_stay_days,float64,0,0.000000,59
3,total_claim_cost,float64,0,0.000000,1026
2,base_encounter_cost,float64,0,0.000000,2
6,insurance_coverage_pct,float64,0,0.000000,553
4,payer_coverage,float64,0,0.000000,534
7,procedure_count,float64,0,0.000000,30
8,prior_inpatient_visits,int64,0,0.000000,63


## Outcome and time coverage

In [17]:
pd.DataFrame({
    "rows": [len(df)],
    "patients": [df["patient"].nunique()],
    "positive_outcomes": [int(df["readmitted_30d"].sum())],
    "readmission_rate": [df["readmitted_30d"].mean()],
    "min_start": [df["start"].min()],
    "max_start": [df["start"].max()],
})

,rows,patients,positive_outcomes,readmission_rate,min_start,max_start
0,1131,153,409,0.361627,2011-01-07 22:39:45+00:00,2021-12-22 21:00:58+00:00


## Basic integrity checks

In [18]:
assert df["readmitted_30d"].isin([0, 1]).all()
assert df["length_of_stay_days"].ge(0).all()
assert df["encounter_id"].notna().all()
assert df["patient"].notna().all()

# The current encounter itself must never count as a prior visit.
assert df["prior_inpatient_visits"].ge(0).all()

print("Integrity checks passed.")

Integrity checks passed.


## Export modeling dataset

The exported file contains identifiers and timestamps for traceability plus all selected predictors and the target.

Identifiers are **not** included as model features.

In [19]:
export_cols = [
    c for c in [
        "patient", "encounter_id", "start", "stop",
        *features,
        "readmitted_30d",
    ]
    if c in df.columns
]

model_df = df[export_cols].copy()

output_path = PROCESSED_DATA_DIR / "readmission_model_data.csv"
model_df.to_csv(output_path, index=False)

print("Saved: data/processed/readmission_model_data.csv")
print(f"Shape: {model_df.shape}")
model_df.head()

Saved: data/processed/readmission_model_data.csv
Shape: (1131, 25)


,patient,encounter_id,start,stop,age,length_of_stay_days,base_encounter_cost,total_claim_cost,payer_coverage,patient_cost,...,days_since_previous_discharge_missing,admission_month,admission_dayofweek,weekend_admission,gender,race,ethnicity,marital,payer_name,readmitted_30d
0,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,cc36dc26-6019-94a5-93b7-7bbf312e6fc4,2011-01-07 22:39:45+00:00,2011-01-08 22:39:45+00:00,81.0,1.0,87.71,19437.88,15454.3,3983.58,...,1,1,4,0,M,white,nonhispanic,S,Medicare,1
1,8b072645-9aae-9166-11e4-45e0e6bf7618,a7775d8f-12d3-356f-7fd5-34412f3870df,2011-01-18 13:31:14+00:00,2011-01-19 13:31:14+00:00,64.0,1.0,87.71,19044.41,0.0,19044.41,...,1,1,1,0,F,white,hispanic,M,NO_INSURANCE,1
2,fa53fdfe-9fb4-a535-b49b-3ce0fd0998cd,82b2cdd6-354a-88ab-adbb-058090672d60,2011-01-19 21:16:09+00:00,2011-01-20 21:16:09+00:00,39.0,1.0,87.71,0.00,0.0,0.00,...,1,1,2,0,F,white,nonhispanic,S,Medicaid,0
3,a80b1160-93f0-db7e-9f23-04ea6fdddfaf,914cbd5c-b56a-9ae9-319e-15c719364ffd,2011-01-23 03:42:18+00:00,2011-01-24 03:42:18+00:00,82.0,1.0,87.71,8260.15,0.0,8260.15,...,1,1,6,1,M,white,nonhispanic,M,NO_INSURANCE,1
4,01274098-150f-8211-6150-29f2a2da266c,9078cc7d-26ca-b1a6-98c2-0dbe6a9b84d2,2011-01-23 14:53:31+00:00,2011-01-24 14:53:31+00:00,81.0,1.0,146.18,0.00,0.0,0.00,...,1,1,6,1,F,white,nonhispanic,M,Dual Eligible,0


## Final Feature Notes

The final modeling table contains **1,132 eligible inpatient encounters** and combines information from the patient, encounter, payer, and procedure tables.

## Target Variable

The target variable is:

`readmitted_30d`

This variable equals **1** when the patient's next inpatient admission occurs between 0 and 30 days after discharge and **0** otherwise.

Encounters without sufficient follow-up time are excluded before modeling.

## Final Predictor Groups

### Patient Characteristics

- `age_at_admission`
- `gender`
- `race`
- `ethnicity`
- `marital`

Age is calculated at the time of each admission rather than using a fixed patient age.

Demographic variables are retained for analytical comparison but should be interpreted cautiously. Associations observed in this synthetic dataset should not be interpreted as causal relationships.

### Current Encounter Characteristics

- `length_of_stay_days`
- `base_encounter_cost`
- `total_claim_cost`
- `payer_coverage`
- `procedure_count`

These features describe the current inpatient encounter.

A timing assumption is important for these variables. Length of stay, procedure count, total claim cost, and payer coverage are most appropriate when the model is interpreted as a **discharge-time or post-encounter model**. These variables would not all be known at the beginning of an admission.

If the intended use changes to prediction at admission time, these variables should be removed or replaced with information available before or at admission.

### Prior-Utilization Characteristics

- `prior_inpatient_visits`
- `prior_readmissions`
- `days_since_previous_discharge`
- `missing_prior_history`

These variables summarize historical utilization prior to the current encounter.

To reduce target leakage, prior readmission counts are calculated using only earlier encounters. The current encounter's outcome is not included in its own historical feature values.

`days_since_previous_discharge` is missing for approximately **13.5% of eligible encounters** because these observations represent patients without a previous inpatient encounter in the available history.

This missingness is expected rather than necessarily erroneous. The associated `missing_prior_history` indicator explicitly identifies these cases, while numerical preprocessing can impute the missing interval value.

### Calendar Characteristics

- `admission_month`
- `admission_dayofweek`
- `admission_weekend`

These features allow the models to identify recurring temporal patterns without using the exact admission date as a direct predictor.

### Insurance Characteristic

- `payer_name`

Payer identifiers are converted to payer names for interpretability before categorical encoding.

## Missingness in Final Model Features

Most final model features are complete.

The primary missing predictor is:

- `days_since_previous_discharge`: approximately **13.5% missing**

This occurs primarily when no previous inpatient encounter exists for the patient in the available history.

Marital status contains less than **0.1% missingness**.

Other selected predictors are effectively complete in the final modeling table.

## Leakage Controls

Several controls are used to reduce information leakage:

1. Encounters are ordered chronologically.
2. Future admissions are used only to construct the target.
3. Historical utilization features use information from earlier encounters only.
4. Encounters without a complete 30-day follow-up period are excluded.
5. Training and testing are separated chronologically.
6. Preprocessing is fitted using the training data and then applied to the test data.

## Feature Engineering Conclusion

The final feature table combines demographic characteristics, current encounter information, prior utilization, calendar variables, procedures, costs, and payer information while preserving the chronological structure of the data.

The most important analytical consideration is the **prediction point**. Several current-encounter variables, including length of stay, procedure count, total claim cost, and payer coverage, become fully known only during or after hospitalization.

For this reason, the current model is best described as estimating **30-day readmission risk using information available by the end of the index inpatient encounter**, rather than as a model that makes predictions strictly at the time of admission.

A future extension will create a separate admission-time model using only information available before or at the start of the inpatient encounter and compare its performance with the current model.
